# 01 趋势策略研究示例

演示 QIS 平台的研究流程：读取缓存行情 → 换月调整 → 生成权重 → 回测 → 绩效与归因。

前提：已运行 `uv run qis fetch-data` 建好本地缓存。

In [ ]:
import pandas as pd
from qis.data.store import DataStore
from qis.data.universe import Universe
from qis.cli import _adjusted_price_matrix

u = Universe.from_yaml()
store = DataStore()

# 换月调整后的价格矩阵 + 换月日掩码 + 换月识别体检。
# 这里**先不切窗口**：信号要用完整历史算，否则回测窗口的第一年会被 lookback 预热吃掉。
full, roll_mask, diag = _adjusted_price_matrix(store, u, with_roll_mask=True, with_diagnostics=True)
print(full.shape)

# 换月识别不可信的标的先看一眼——它们的收益没被可靠调整
suspect = {n: d for n, d in diag.items() if not d["plausible"]}
print(f"换月识别存疑 {len(suspect)}/{len(diag)}: {sorted(suspect)}")
full.tail(3)


In [ ]:
from qis.data.panel import to_returns
from qis.strategy.trend import trend_weights
from qis.portfolio.construction import cap_binding_share, vol_target_factor, weight_band
from qis.backtest.costs import load_settings

settings = load_settings(); bt = settings["backtest"]

# 收益一律走 to_returns：价格矩阵是多市场日历的并集，标的假期是 NaN 洞，
# 直接 pct_change 会把跨假期的涨跌整段丢掉。
rets = to_returns(full)

w_raw = trend_weights(full)                      # 策略权重（毛敞口=1）
scale, raw = vol_target_factor(w_raw, rets, target=bt["vol_target"],
                               span=bt["vol_lookback"], ann_factor=bt["ann_factor"],
                               max_leverage=bt["max_leverage"])
w = w_raw.mul(scale, axis=0).where(scale.notna(), w_raw)
w = weight_band(w, bt["rebal_band"])             # 无交易带作用在实际持仓上
w.tail(3)


In [ ]:
from qis.backtest.costs import cost_bps_by_name
from qis.backtest.engine import run_backtest
from qis.analytics.metrics import summary

# 信号都算完了，现在才切回测窗口
START = "2010-01-01"
prices = full.loc[START:].dropna(how="all")
wb = w.reindex(index=prices.index, columns=prices.columns).fillna(0.0)
mask = roll_mask.reindex(index=prices.index, columns=prices.columns).fillna(False)

cost = cost_bps_by_name(u.asset_classes(), settings["cost_bps"])
res = run_backtest(prices, wb, cost_bps=cost, roll_mask=mask)
stats = summary(res.net, res.turnover)           # ann_factor 从索引推断

# 杠杆上限长期绑定 = 波动目标没生效，调 vol_target 不会有反应
cap = cap_binding_share(raw, bt["max_leverage"], prices.index)
print(f"杠杆上限({bt['max_leverage']:g}×)绑定比例: {cap:.0%}  实现波动: {stats['ann_vol']:.2%}"
      f"  目标: {bt['vol_target']:.0%}")
stats.round(3)


In [ ]:
import matplotlib.pyplot as plt
from qis.analytics.report import tearsheet

tearsheet(res.net, res.turnover, title="trend demo")
plt.show()

In [ ]:
# 分标的年化盈亏归因：复用引擎算好的收益（res.returns），不要重算
contrib = ((res.weights * res.returns).mean() * stats["ann_factor"]).sort_values()
contrib.plot.barh(figsize=(8, 6), title="Annual P&L contribution by instrument")
plt.show()
